# 문자열과 날짜, 쓸 수 있는 형태로

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🔤 문자열과 날짜, 쓸 수 있는 형태로 — 데이터 랭글링

### — 지저분한 텍스트와 뒤섞인 날짜를, 분석 가능한 데이터로 바꾸는 법 —

---

## 📋 학습 목표

이 자습서를 마치면 여러분은 다음을 할 수 있습니다.

1. 문자열 오염의 네 가지 유형(공백·대소문자·표기 혼재·숨은 패턴)을 진단하고 `str` 기능으로 정제할 수 있습니다.
2. 단순 문자열 기능과 정규표현식(Regular Expression) 중 무엇을 써야 할지 판단하고, 패턴을 뽑아내는 식을 작성할 수 있습니다.
3. 여러 포맷이 뒤섞인 날짜 문자열을 확인하고, 알맞은 파싱 전략으로 진짜 날짜(datetime)로 바꿀 수 있습니다.
4. `dt` 기능으로 날짜에서 연·월·요일·시각 같은 부품을 꺼내 분석용 컬럼을 만들 수 있습니다.
5. 분석 목적에 맞는 시간 단위를 골라 리샘플링(resample)·이동평균(rolling)으로 추세를 읽을 수 있습니다.
6. 원본 로그 파일에서 날짜·IP·요청 경로를 파싱해 집계 테이블을 만들 수 있습니다.
7. 데이터를 처리하기 전에 원본의 패턴을 먼저 눈으로 확인하는 습관을 들일 수 있습니다.

> 💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해보세요. 자습서를 마친 뒤 다시 돌아와 비교하면 여러분의 성장이 눈에 보일 거예요.

## 📚 목차

| Part | 내용 | 핵심 질문 |
| --- | --- | --- |
| Part 0 | 분석 상황과 학습 지도 | 오늘 우리는 어떤 '텍스트'와 '날짜'를 마주할까요? |
| Part 1 | 문자열 정제 — `str` 기초 | 사람 눈엔 같은데 컴퓨터엔 다른 값, 어떻게 맞출까요? |
| Part 2 | 문자열 검색·분리·추출 | 텍스트 안에서 원하는 조각만 어떻게 골라낼까요? |
| Part 3 | 정규표현식 | 규칙은 있지만 자리가 제각각인 패턴을 어떻게 뽑을까요? |
| Part 4 | 날짜 파싱(datetime) | 뒤섞인 날짜 문자열을 어떻게 진짜 날짜로 바꿀까요? |
| Part 5 | `dt` 기능 — 날짜의 부품 | 날짜에서 연·월·요일·시각을 어떻게 꺼낼까요? |
| Part 6 | 리샘플링·이동평균 | 분석 목적에 맞는 시간 단위로 어떻게 묶을까요? |
| 종합 실습 | 웹 로그 파싱 | 한 줄짜리 로그에서 표를 만들어낼 수 있을까요? |
| 정리 | 핵심 요약 · 다음 시간 | 오늘 배운 것이 내일 무엇으로 이어질까요? |

## 분석 상황과 학습 지도

# 0. 분석 상황과 학습 지도

## 지난 시간에는 무엇을 했나요?

바로 전 시간에는 **흩어진 테이블을 하나로 합치는** 일을 했습니다. 주문(orders)·고객(customers)·상품(products) 테이블을 `merge`로 연결하고, `groupby`로 묶어 월별 매출 요약표를 만들었죠. 여러 표가 비로소 하나의 인사이트로 모였습니다.

그런데 테이블을 합치다 보면 자꾸 걸리는 게 있었습니다. 같은 지역인데 `'서울'`과 `'Seoul'`이 따로 세어지고, 날짜 컬럼은 분명 날짜인데 계산이 안 됐죠. **합치는 데는 성공했지만, 그 안의 글자와 날짜는 아직 "쓸 수 있는 형태"가 아니었던 것**입니다.

## 오늘의 여정

오늘은 데이터 안의 **문자열(text)과 날짜(datetime)** 를 분석 가능한 형태로 다듬습니다. 사람에게는 다 읽히지만 컴퓨터에게는 제각각인 텍스트와, 형식이 뒤섞인 날짜를 정리하는 기술을 순서대로 배웁니다.

```text
[Part 1] 문자열 정제(str)     →  공백·대소문자·표기를 깔끔하게 통일
   ↓
[Part 2] 검색·분리·추출        →  텍스트 안에서 원하는 조각 골라내기
   ↓
[Part 3] 정규표현식(regex)     →  자리가 제각각인 패턴을 규칙으로 추출
   ↓
[Part 4] 날짜 파싱(datetime)   →  뒤섞인 날짜 문자열을 진짜 날짜로
   ↓
[Part 5] dt 기능               →  날짜에서 연·월·요일·시각 꺼내기
   ↓
[Part 6] 리샘플링·이동평균      →  목적에 맞는 시간 단위로 묶어 추세 읽기
   ↓
[종합 실습] 웹 로그 파싱        →  한 줄짜리 로그를 집계 테이블로
```

## 이 자습서 사용법

이 노트북은 **혼자서도 끝까지 학습할 수 있도록** 만들었습니다. 다음 네 박자로 진행하세요.

- 📖 **읽고** — 개념 설명을 천천히 읽습니다.
- 💻 **실행하고** — 코드 셀을 위에서부터 순서대로 실행합니다. (단축키: `Shift + Enter`)
- ✏️ **고쳐보고** — "스스로 해보자!" 칸에서 코드를 직접 바꿔봅니다.
- 🤔 **답해보고** — 체크포인트 질문에 스스로 답해봅니다. 틀려도 괜찮습니다.

> ⚠ **주의:** 코드 셀은 반드시 **위에서부터 순서대로** 실행해야 합니다. 중간 셀을 건너뛰면 "앞에서 만든 변수가 없다"는 오류가 날 수 있어요. 오류는 잘못이 아니라 디버깅의 단서입니다. 천천히 읽어가세요.

## 오늘의 무대: 다시, 가상의 이커머스 "모두마켓"

이번 모듈 내내 우리는 가상의 온라인 쇼핑몰 **모두마켓**을 분석하고 있습니다. 오늘 여러분에게는 새로운 미션이 주어졌습니다.

> "고객 연락처랑 가입일 좀 정리해 주실래요? 그리고 주문 시각 기준으로 **요일별·시간대별 매출 추세**도 보고 싶어요. 그런데… 데이터가 좀 제멋대로예요."

동료의 말 그대로입니다. 아래 셀을 실행하면 만들어지는 데이터에는 **현장에서 흔히 보는 텍스트·날짜 오염**이 일부러 심어져 있습니다.

- 이메일에 대소문자·앞뒤 공백이 섞여 있고
- 전화번호 형식이 `010-1234-5678` / `01012345678` / `010.1234.5678` 처럼 제각각이며
- 가입일·주문일이 `2024-03-15` / `2024/03/15` / `20240315` 처럼 포맷이 뒤죽박죽입니다.

> **🎯 오늘의 핵심**
> **처리하기 전에 원본의 패턴부터 눈으로 확인합니다.** 문자열·날짜 정제는 "어떻게 생겼는지"를 먼저 본 다음에야 올바른 방법을 고를 수 있습니다.

> 📌 **어제와 오늘의 `customers`가 왜 다른가요? — 같은 테이블의 다른 뷰**
> 어제(D+004)의 `customers`는 `age`·`gender`·`membership` 중심이었지만, 오늘은 `name`·`email`·`phone`·`signup_date` 중심입니다. **데이터셋이 바뀐 것이 아닙니다.** 같은 모두마켓의 customers 테이블이지만 *분석 목적에 따라 다른 컬럼을 본다*는 뜻이에요. 어제는 매출·세그먼트 분석에 필요한 *인구통계 뷰*를, 오늘은 마케팅 발송·시계열 가입자 흐름 분석에 필요한 *연락처·가입일 뷰*를 꺼내 옵니다. 실무에서 흔히 보는 *"같은 테이블, 다른 뷰"* 상황이에요.

> **읽는 법:** 벌써 여러 가지가 눈에 띕니다. `email`에 대문자와 앞뒤 공백이 보이고(`repr`로 보면 더 분명합니다), `phone`은 줄마다 형식이 다르며, `signup_date`·`order_datetime`은 `2024-03-15`와 `2024/03/15`처럼 구분자가 섞여 있습니다. `product_name`에는 보이지 않는 공백도 숨어 있을 수 있어요.

사람 눈에는 "다 비슷한데 뭐가 문제지?" 싶지만, 컴퓨터에게는 **공백 하나, 대문자 하나가 완전히 다른 값**입니다. 그래서 집계가 어긋나고 날짜 계산이 막히죠. 이제 이 텍스트와 날짜를 하나씩 "쓸 수 있는 형태"로 바꿔봅시다. 출발점은 가장 흔한 오염, **문자열**입니다.

In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False   # 마이너스 부호 깨짐 방지
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [7]:
# ─────────────────────────────────────────────
# 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 데이터가 모두 준비됩니다.
# (실제 현장처럼 '텍스트·날짜 오염'을 일부러 심어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers) -------------------------------------------------
n_customers = 200
last_names = ["김", "이", "박", "최", "정", "강", "조", "윤", "장", "임"]
first_names = ["민준", "서연", "도윤", "지우", "하준", "서윤", "예준", "지유", "주원", "지호"]
names = [np.random.choice(last_names) + np.random.choice(first_names) for _ in range(n_customers)]
# 이름 일부에 앞뒤 공백 오염
for i in np.random.choice(n_customers, 15, replace=False):
    names[i] = "  " + names[i] + " "

# 이메일: 대소문자 + 앞뒤 공백 오염
domains = ["gmail.com", "naver.com", "MODUMARKET.com", "daum.net", "kakao.com"]
emails = []
for i in range(n_customers):
    e = f"user{i+1:03d}@{np.random.choice(domains)}"
    r = np.random.random()
    if r < 0.20:
        e = e.upper()              # 전체 대문자
    elif r < 0.35:
        e = "  " + e + " "         # 앞뒤 공백
    emails.append(e)

# 전화번호: 네 가지 형식 혼재
def make_phone(i):
    mid, end = np.random.randint(1000, 9999), np.random.randint(1000, 9999)
    fmt = i % 4
    if fmt == 0:
        return f"010-{mid}-{end}"
    if fmt == 1:
        return f"010{mid}{end}"
    if fmt == 2:
        return f"010.{mid}.{end}"
    return f"+82 10 {mid} {end}"
phones = [make_phone(i) for i in range(n_customers)]

# 지역: 표기 혼재(공백·영문)
region = np.random.choice(
    ["서울", " 서울 ", "Seoul", "경기", "부산", "인천", "대구"],
    n_customers, p=[0.30, 0.05, 0.05, 0.25, 0.15, 0.10, 0.10])

# 가입일: 네 가지 날짜 포맷 혼재(문자열로 저장)
signup_base = pd.to_datetime("2024-01-01") + pd.to_timedelta(np.random.randint(0, 365, n_customers), unit="D")
signup = []
for i, d in enumerate(signup_base):
    f = i % 4
    if f == 0:
        signup.append(d.strftime("%Y-%m-%d"))
    elif f == 1:
        signup.append(d.strftime("%Y/%m/%d"))
    elif f == 2:
        signup.append(d.strftime("%Y.%m.%d"))
    else:
        signup.append(d.strftime("%Y%m%d"))

customers = pd.DataFrame({
    "customer_id": [f"C{i+1:04d}" for i in range(n_customers)],
    "name": names,
    "email": emails,
    "phone": phones,
    "region": region,
    "signup_date": signup,
})

# 2) 주문(orders) ----------------------------------------------------
catalog = [("무선 이어폰", "P-1001", 89000), ("블루투스 스피커", "P-1002", 49000),
           ("노트북 거치대", "P-1003", 29000), ("기계식 키보드", "P-1004", 119000),
           ("USB-C 충전기", "P-1005", 19000), ("보조배터리", "P-1006", 39000),
           ("스마트워치", "P-1007", 159000), ("액션캠", "P-1008", 229000)]
n_orders = 2000
pick = np.random.randint(0, len(catalog), n_orders)

product_name, amount = [], []
for j in pick:
    name, codeid, price = catalog[j]
    label = f"{name} ({codeid})"
    r = np.random.random()
    if r < 0.18:
        label = "  " + label + "  "        # 앞뒤 공백
    elif r < 0.30:
        label = label.replace(" ", "  ")    # 단어 사이 이중 공백
    product_name.append(label)
    amount.append(price * np.random.choice([1, 1, 1, 2, 3]))

coupon = np.random.choice(
    ["SALE2025", "WELCOME10", "VIP-2025", "summer25", "FREESHIP", ""],
    n_orders, p=[0.18, 0.15, 0.10, 0.10, 0.07, 0.40])

channel = np.random.choice(["web", "app", "app ", "APP"], n_orders, p=[0.45, 0.45, 0.05, 0.05])

# 주문 일시: 두 가지 포맷 혼재 + 소수의 오류값
dt_base = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 90 * 24 * 3600, n_orders), unit="s")
order_datetime = []
for i, d in enumerate(dt_base):
    if i % 2 == 0:
        order_datetime.append(d.strftime("%Y-%m-%d %H:%M:%S"))
    else:
        order_datetime.append(d.strftime("%Y/%m/%d %H:%M:%S"))
# 오류값 심기: 파싱 불가능한 문자열 12건
for i in np.random.choice(n_orders, 12, replace=False):
    order_datetime[i] = np.random.choice(["처리중", "-", "unknown"])

orders = pd.DataFrame({
    "order_id": [f"O{i+1:05d}" for i in range(n_orders)],
    "customer_id": np.random.choice(customers["customer_id"], n_orders),
    "product_name": product_name,
    "amount": amount,
    "coupon": coupon,
    "channel": channel,
    "order_datetime": order_datetime,
})

print("모두마켓 데이터 생성 완료")
print("customers:", customers.shape, "| orders:", orders.shape)

print(customers)

모두마켓 데이터 생성 완료
customers: (200, 6) | orders: (2000, 7)
    customer_id name                 email             phone region  \
0         C0001  조지우     user001@daum.net      010-8973-9577     경기   
1         C0002  윤하준    user002@gmail.com        01090709666     서울   
2         C0003  조지호     USER003@NAVER.COM     010.9609.7490     서울   
3         C0004  박예준     USER004@KAKAO.COM  +82 10 4600 7783     인천   
4         C0005  윤하준     USER005@KAKAO.COM     010-5971-2205     대구   
..          ...  ...                   ...               ...    ...   
195       C0196  장하준     user196@kakao.com  +82 10 1571 3382     경기   
196       C0197  김지호    user197@kakao.com      010-6084-1277  Seoul   
197       C0198  임민준      user198@daum.net       01080592887     서울   
198       C0199  이서윤    user199@naver.com      010.2265.8491     서울   
199       C0200  장지유      user200@daum.net  +82 10 2040 9073     인천   

    signup_date  
0    2024-04-27  
1    2024/09/26  
2    2024.03.20  
3      20240402  
4 

NameError: name 'df' is not defined

In [8]:
# 데이터가 어떻게 '오염'되어 있는지 맨 위 몇 줄을 직접 봅니다.
print("=== customers (고객) ===")
display(customers.head())
print("\n=== orders (주문) ===")
display(orders.head())

=== customers (고객) ===


,customer_id,name,email,phone,region,signup_date
0,C0001,조지우,user001@daum.net,010-8973-9577,경기,2024-04-27
1,C0002,윤하준,user002@gmail.com,01090709666,서울,2024/09/26
2,C0003,조지호,USER003@NAVER.COM,010.9609.7490,서울,2024.03.20
3,C0004,박예준,USER004@KAKAO.COM,+82 10 4600 7783,인천,20240402
4,C0005,윤하준,USER005@KAKAO.COM,010-5971-2205,대구,2024-09-22



=== orders (주문) ===


,order_id,customer_id,product_name,amount,coupon,channel,order_datetime
0,O00001,C0131,무선 이어폰 (P-1001),89000,FREESHIP,app,2025-01-01 23:12:56
1,O00002,C0094,USB-C 충전기 (P-1005),57000,WELCOME10,web,2025/01/25 12:15:09
2,O00003,C0008,기계식 키보드 (P-1004),357000,,web,2025-01-31 14:14:25
3,O00004,C0086,블루투스 스피커 (P-1002),147000,,app,2025/03/30 02:12:54
4,O00005,C0089,무선 이어폰 (P-1001),89000,,web,2025-02-18 23:06:24


## 문자열 정제 — `str` 기초

# 1. 문자열 정제 — `str` 기초

Part 0에서 `email`과 `region`을 봤습니다. ` 서울 `(앞뒤 공백)과 `Seoul`(영문), `서울`이 섞여 있었죠. 만약 `region`별 고객 수를 세면 어떻게 될까요? `'서울'`과 `' 서울 '`이 **다른 지역으로 따로 집계**됩니다. 사람 눈엔 같은 서울인데 말이죠.

> ❓ **이 파트에서 답할 질문:** 사람 눈에는 같지만 컴퓨터에는 다른 값(공백·대소문자 차이)을, 어떻게 하나로 맞출까요?

## 💡 쉽게 말하면 — 글자에도 '보이지 않는 군더더기'가 있다

문자열(string)은 글자들이 줄지어 있는 값입니다. 그런데 우리 눈에 안 보이는 군더더기가 자주 끼어 있어요.

```text
원본:   "  Seoul "      ← 앞에 공백 2칸, 뒤에 공백 1칸, 게다가 영문 대문자 S
정제:   "seoul"  →(매핑)→  "서울"

"app"  vs  "app "  vs  "APP"   ← 사람에겐 같은 앱, 컴퓨터에겐 서로 다른 3개
```

정제의 기본기는 세 가지입니다. **공백 제거(strip) → 대소문자 통일(lower/upper) → 표기 치환(replace)**. 이 순서로 다듬으면 대부분의 표기 혼재가 사라집니다.

## 자세히 알아보기 — pandas의 `.str` 접근자(accessor)

pandas에서 문자열 컬럼(Series)을 다룰 때는 **`.str`** 을 붙입니다. 그러면 파이썬 문자열 기능을 **컬럼 전체에 한 번에** 적용할 수 있어요. (Part 0에서 배운 벡터화의 문자열 버전이라고 생각하면 됩니다.)

| 기능 | 하는 일 | 예시 |
| --- | --- | --- |
| `.str.strip()` | 앞뒤 공백 제거 | `" 서울 "` → `"서울"` |
| `.str.lower()` / `.str.upper()` | 소문자/대문자로 통일 | `"APP"` → `"app"` |
| `.str.replace(a, b)` | a를 b로 치환 | `"Seoul"` → `"서울"` |
| `.str.len()` | 글자 수 | `"서울"` → `2` |

> 💡 **핵심:** `.str.strip().str.lower()` 처럼 **점(.)으로 계속 이어 붙일 수 있습니다(method chaining).** 왼쪽부터 차례로 적용돼요. 이 "이어 붙이기"는 다음 시간(D+006)에 전처리 파이프라인으로 발전합니다.

> **읽는 법:** `' 서울 '`과 `'서울'`, `'Seoul'`이 따로따로 세어집니다. `repr()`로 출력하면 `' 서울 '`처럼 따옴표 안의 공백이 눈에 보여요. **정제 전에 `repr`로 원본을 들여다보는 습관**이 오늘의 핵심 태도입니다.

> **읽는 법:** 세 갈래로 갈라졌던 서울이 하나로 합쳐졌습니다. `.str.strip()`이 공백을, `.str.replace()`가 영문 표기를 정리했죠. 이제야 "서울 고객이 몇 명인가"라는 질문에 올바로 답할 수 있습니다.

## 데이터로 확인해 봅시다

- **집계 전 정제는 필수**입니다. `groupby("region")`(D+004) 전에 `region`을 정제하지 않으면, 같은 값이 쪼개져 매출·고객 수가 틀어집니다. 즉 **정제는 올바른 집계의 전제**예요.
- 이메일·아이디는 보통 **소문자로 통일**해 저장합니다. 로그인 비교나 중복 제거(`drop_duplicates`)가 정확해지기 때문입니다.

> 📌 **다른 산업에서는?** 표기 통일은 어디서나 똑같이 중요합니다. 금융은 고객명·주소의 공백/대소문자를 맞춰 **동일인 식별**에 쓰고, 마케팅은 캠페인 태그(`Summer`/`summer`)를 통일해 **채널 성과 집계**를 바로잡으며, 헬스케어는 약품명 표기를 표준화해 **처방 통계**를 정확히 냅니다.

### 스스로 해보자! ✏️ (1)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

`orders`의 `channel` 컬럼에는 `'web'`, `'app'`, `'app '`(뒤 공백), `'APP'`(대문자)이 섞여 있습니다.

1. **(따라하기)** `channel`을 그냥 `value_counts()`로 세어, 같은 앱이 어떻게 쪼개지는지 확인하세요.
2. **(응용)** `.str.strip().str.lower()`로 정제한 `channel_clean` 컬럼을 만들고 다시 세어보세요.
3. **(생각해보기)** 만약 정제하지 않은 채 "앱 주문 비율"을 보고했다면, 어떤 잘못된 결론이 나올까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
print(orders["channel"].value_counts())          # web / app / 'app ' / 'APP' 로 쪼개짐

orders["channel_clean"] = orders["channel"].str.strip().str.lower()
print(orders["channel_clean"].value_counts())     # web / app 두 개로 정리됨
```

정제 전에는 `'app '`과 `'APP'`이 따로 세어져 **앱 주문 수가 실제보다 적게** 잡힙니다. 정제하지 않으면 "앱보다 웹이 많다" 같은 잘못된 결론에 이를 수 있어요.

</details>

### ✅ 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. `.str.strip()`, `.str.lower()`, `.str.replace()`는 각각 어떤 오염을 정리하나요?
2. 정제 전에 `repr()`로 원본을 보는 이유는 무엇인가요?
3. `.str.strip().str.lower()`처럼 점으로 이어 붙이면 적용 순서는 어떻게 되나요?

> 💡 **다음 Part 예고:** 지금은 글자를 *통째로* 다듬었습니다. 그런데 "이메일에서 도메인만", "전화번호에서 숫자만"처럼 **텍스트 안의 일부 조각**만 필요할 때가 있죠. 다음 Part에서는 텍스트를 검색하고, 쪼개고, 일부를 골라내는 법을 배웁니다.

In [9]:
# 예제: 정제 전 — region을 그냥 세면 같은 서울이 쪼개진다
print("[정제 전] region 값의 종류")
print(customers["region"].value_counts())
# repr로 보면 숨은 공백이 보입니다.
print("\n숨은 공백 확인:", [repr(x) for x in customers["region"].unique()])

[정제 전] region 값의 종류
region
경기       61
서울       52
부산       32
인천       19
대구       18
Seoul    10
 서울       8
Name: count, dtype: int64

숨은 공백 확인: ["'경기'", "'서울'", "'인천'", "'대구'", "'부산'", "'Seoul'", "' 서울 '"]


In [10]:
# 예제: strip → lower → replace 3단계로 region 통일하기
clean_region = (
    customers["region"]
    .str.strip()                      # 1) 앞뒤 공백 제거
    .str.replace("Seoul", "서울")      # 2) 영문 표기를 한글로 치환
)
customers["region_clean"] = clean_region

print("[정제 후] region_clean 값의 종류")
print(customers["region_clean"].value_counts())

[정제 후] region_clean 값의 종류
region_clean
서울    70
경기    61
부산    32
인천    19
대구    18
Name: count, dtype: int64


In [11]:
# 예제: 이메일 정제 — 앞뒤 공백 제거 + 소문자 통일
print("[정제 전] 이메일 일부(repr로 공백 노출)")
print([repr(x) for x in customers["email"].head(6)])

customers["email_clean"] = customers["email"].str.strip().str.lower()

print("\n[정제 후]")
print(customers["email_clean"].head(6).tolist())

[정제 전] 이메일 일부(repr로 공백 노출)
["'  user001@daum.net '", "'  user002@gmail.com '", "'USER003@NAVER.COM'", "'USER004@KAKAO.COM'", "'USER005@KAKAO.COM'", "'user006@kakao.com'"]

[정제 후]
['user001@daum.net', 'user002@gmail.com', 'user003@naver.com', 'user004@kakao.com', 'user005@kakao.com', 'user006@kakao.com']


In [ ]:
# 스스로 해보자! (1)
# 아래 주석(#)을 지우고 빈칸(___)을 채운 뒤 실행해보세요.

# print("[정제 전]")
# print(orders["channel"].value_counts())

# orders["channel_clean"] = orders["channel"].str.____().str.____()   # strip, lower
# print("\n[정제 후]")
# print(orders["channel_clean"].value_counts())

## 문자열 검색·분리·추출 — 원하는 조각만 골라내기

# 2. 문자열 검색·분리·추출 — 원하는 조각만 골라내기

Part 1에서 이메일을 소문자로 통일했습니다. 그런데 마케팅 팀이 "**도메인별(gmail/naver…) 고객 분포**가 궁금하다"고 합니다. 이메일 전체가 아니라 `@` 뒤의 **도메인 조각**만 필요하죠. 통째로 다듬는 것만으로는 부족하고, 이제 **텍스트 안에서 원하는 부분만 꺼내는** 기술이 필요합니다.

> ❓ **이 파트에서 답할 질문:** 텍스트 전체가 아니라, 그 안의 특정 조각만 어떻게 검색하고 골라낼까요?

## 💡 쉽게 말하면 — 문자열은 '글자들의 줄'

문자열은 글자가 순서대로 늘어선 줄입니다. 그래서 "포함하는가?", "어디서 끊을까?", "몇 번째 칸?" 같은 질문을 던질 수 있어요.

```text
"user001@gmail.com"
            ▲
          @ 기준으로 쪼개기(split)
   ┌────────┴────────┐
"user001"        "gmail.com"   ← [1]번째 조각이 도메인
```

## 자세히 알아보기 — 검색·분리·추출 도구

| 기능 | 하는 일 | 결과 예시 |
| --- | --- | --- |
| `.str.contains("gmail")` | 그 글자를 포함하는가? (참/거짓) | `True`/`False` |
| `.str.startswith("010")` | 그 글자로 시작하는가? | `True`/`False` |
| `.str.split("@")` | 기준 글자로 쪼개 리스트로 | `["user001", "gmail.com"]` |
| `.str.split("@").str[1]` | 쪼갠 뒤 N번째 조각 선택 | `"gmail.com"` |
| `.str[0:3]` | 글자 위치로 잘라내기(슬라이싱) | `"010"` |

> 💡 **개념 연결:** `.str.contains(...)`의 결과는 참/거짓의 **불리언(boolean) Series**입니다. D+003에서 배운 불리언 필터링(`df[조건]`)에 그대로 넣을 수 있어요. "조건 → 불리언 → 필터"라는 흐름은 문자열에서도 똑같이 작동합니다.

> **읽는 법:** `.str.contains("gmail")`은 각 행이 `gmail`을 포함하는지 참/거짓으로 알려주고, 그 참/거짓 줄을 `customers[...]`에 넣으면 조건을 만족하는 행만 남습니다. 문자열 검색과 D+003의 필터링이 자연스럽게 만나는 장면이에요.

> **읽는 법:** `.str.split("@")`은 `"user001@gmail.com"`을 `["user001", "gmail.com"]`로 쪼갭니다. 여기서 `.str[1]`로 두 번째 조각(도메인)만 골랐어요. 이제 마케팅 팀이 원하던 도메인별 분포가 한눈에 보입니다.

> **읽는 법:** `.str[0:3]`은 앞 3글자를 잘라냅니다. 그런데 형식이 제각각이라 `010`도 있고 `+82`도 있죠. 단순 위치 자르기(슬라이싱)는 **형식이 일정할 때만** 안전합니다. 형식이 뒤섞인 전화번호에서 "숫자만 뽑기"는 위치 자르기로는 어렵습니다 — 바로 다음 Part의 **정규표현식**이 필요한 순간이에요.

## 데이터로 확인해 봅시다

- `.str.contains()`는 **텍스트 필터링의 핵심**입니다. "특정 키워드가 들어간 상품명", "오류 메시지에 'timeout'이 포함된 로그"를 골라낼 때 매일 씁니다.
- `.str.split().str[N]`은 **구분자가 일정한** 데이터(이메일의 `@`, 경로의 `/`)에서 조각을 뽑는 가장 간단한 방법입니다.

> 📌 **다른 산업에서는?** 조각 추출은 분야를 가리지 않습니다. 마케팅은 UTM 파라미터를 `&`로 쪼개 채널을 분리하고, 제조는 제품 시리얼의 앞자리로 공장을 식별하며, IT 운영은 로그 한 줄을 공백으로 쪼개 필드를 나눕니다.

### 스스로 해보자! ✏️ (2)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `customers["email_clean"]`에서 `'naver'`를 포함하는 고객 수를 세어보세요.
2. **(응용)** 도메인(`@` 뒤)이 회사 메일 `'modumarket.com'`인 고객만 필터링해 몇 명인지 출력하세요.
3. **(생각해보기)** 전화번호에서 "순수 숫자 11자리"만 뽑고 싶다면, 왜 `.str.split()`이나 슬라이싱만으로는 어려울까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
print("naver 고객:", customers["email_clean"].str.contains("naver").sum())

customers["domain2"] = customers["email_clean"].str.split("@").str[1]
company = customers[customers["domain2"] == "modumarket.com"]
print("회사 메일 고객 수:", len(company))
```

3번: 전화번호는 `-`, `.`, 공백, `+82` 등 **자리가 제각각**이라, 고정된 위치로 자르거나 한 가지 구분자로 쪼개는 방식이 통하지 않습니다. "숫자라는 규칙은 같지만 자리는 다른" 패턴을 다루려면 정규표현식이 필요합니다.

</details>

### ✅ 짚고 넘어가기

1. `.str.contains()`의 결과는 어떤 자료형이고, 어디에 바로 쓸 수 있나요?
2. `.str.split("@").str[1]`은 무엇을 골라내나요?
3. 단순 슬라이싱(`.str[0:3]`)이 안전하려면 데이터가 어떤 조건을 만족해야 하나요?

> 💡 **다음 Part 예고:** `split`과 슬라이싱은 "구분자나 위치가 일정할 때"만 통합니다. 전화번호처럼 **규칙(숫자)은 있지만 자리가 제각각인** 패턴은요? 다음 Part에서 패턴 자체를 규칙으로 적어 뽑아내는 **정규표현식**을 만납니다.

In [27]:
# 예제: contains — 특정 도메인 고객만 필터링
is_gmail = customers["email_clean"].str.contains("gmail")   # 불리언 Series
print("gmail 사용 고객 수:", is_gmail.sum())

gmail_customers = customers[is_gmail]                       # 불리언 필터링(D+003 복습)
print(gmail_customers[["customer_id", "email_clean"]].head())

gmail 사용 고객 수: 39
   customer_id        email_clean
1        C0002  user002@gmail.com
9        C0010  user010@gmail.com
10       C0011  user011@gmail.com
12       C0013  user013@gmail.com
19       C0020  user020@gmail.com


In [28]:
# 예제: split — @ 기준으로 쪼개 도메인만 추출
domain = customers["email_clean"].str.split("@").str[1]   # 쪼갠 리스트의 [1]번째 = 도메인
customers["domain"] = domain

print("[도메인별 고객 분포]")
print(customers["domain"].value_counts())

[도메인별 고객 분포]
domain
naver.com         46
kakao.com         42
gmail.com         39
modumarket.com    37
daum.net          36
Name: count, dtype: int64


In [33]:
# 예제: 슬라이싱 + startswith — 전화번호 앞 3자리 확인
print("전화번호 앞 3자리(원본 그대로):")
print(customers["phone"].str[0:3].value_counts().head())

# '+82'로 시작하는 국제 표기는 몇 건인가?
intl = customers["phone"].str.startswith("+82")
print("\n'+82' 국제 표기 건수:", intl.sum())

전화번호 앞 3자리(원본 그대로):
phone
010    150
+82     50
Name: count, dtype: int64

'+82' 국제 표기 건수: 50


In [32]:
# 스스로 해보자! (2)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

# 1) naver 포함 고객 수
print("naver 고객:", customers["email_clean"].str.contains("naver").sum())

# 2) 도메인이 modumarket.com 인 고객

customers["domain2"] = customers["email_clean"].str.split("@").str[1]
company = customers[customers["domain2"] == "modumarket.com"]
print("회사 메일 고객 수:", len(company))

print(customers["domain2"])

naver 고객: 46
회사 메일 고객 수: 37
0       daum.net
1      gmail.com
2      naver.com
3      kakao.com
4      kakao.com
         ...    
195    kakao.com
196    kakao.com
197     daum.net
198    naver.com
199     daum.net
Name: domain2, Length: 200, dtype: object


## 정규표현식 — 패턴을 규칙으로 뽑아내기

# 3. 정규표현식 — 패턴을 규칙으로 뽑아내기

Part 2 마지막에서 막혔습니다. 전화번호는 `010-1234-5678`, `01012345678`, `+82 10 1234 5678`처럼 형식이 제각각이라, 위치로 자르거나 한 구분자로 쪼개는 방법이 통하지 않았죠. "숫자라는 규칙은 같은데 자리는 다르다" — 이럴 때 쓰는 도구가 **정규표현식**입니다.

> ❓ **이 파트에서 답할 질문:** 자리는 제각각이지만 규칙은 있는 패턴(숫자·코드·도메인)을, 어떻게 한 번에 뽑아낼까요?

## 💡 쉽게 말하면 — '글자의 종류'를 설명하는 작은 언어

정규표현식(Regular Expression, 줄여서 regex 또는 정규식)은 **"내가 찾는 글자가 어떻게 생겼는지"를 기호로 설명하는 언어**입니다. 정확한 글자 대신 *종류와 개수*로 묘사하죠.

```text
"010-1234-5678" 에서 숫자만 → 0101234 5678
설명: "숫자(0~9)가 한 개 이상 이어진 부분을 다 찾아라"

기호로:   \d+      (\d = 숫자 한 개, + = 한 개 이상 반복)
```

## 자세히 알아보기 — 자주 쓰는 기호와 함수

먼저 패턴을 묘사하는 **기호**입니다. (지금 다 외울 필요 없어요. "이런 게 있다" 정도로 보세요.)

| 기호 | 뜻 | 예 |
| --- | --- | --- |
| `\d` | 숫자 한 개 (0~9) | `\d\d` → `"12"` |
| `\w` | 글자/숫자/밑줄 한 개 | `\w+` → `"user001"` |
| `+` | 바로 앞 패턴을 1번 이상 반복 | `\d+` → `"12345"` |
| `[...]` | 괄호 안 글자 중 하나 | `[A-Z]` → 대문자 한 개 |
| `.` | 아무 글자 한 개 | |
| `()` | 뽑아낼 부분을 묶음(그룹) | `(\d+)` |

다음은 pandas에서 정규식을 쓰는 **함수**입니다.

| 함수 | 하는 일 |
| --- | --- |
| `.str.extract(r"(패턴)")` | 그룹 `()` 부분을 뽑아 새 컬럼으로 |
| `.str.findall(r"패턴")` | 일치하는 모든 조각을 리스트로 |
| `.str.replace(r"패턴", "", regex=True)` | 패턴에 맞는 부분을 치환/삭제 |
| `.str.contains(r"패턴", regex=True)` | 패턴이 있는지 참/거짓 |

> ⚠ **주의:** 정규식 문자열 앞에는 보통 `r"..."` (raw string)을 붙입니다. `\d` 같은 역슬래시 기호가 깨지지 않도록 하는 안전장치예요.

> **읽는 법:** `[^0-9]`는 "숫자가 아닌 글자"를 뜻합니다(`^`가 괄호 안 맨 앞에 오면 '부정'). 그걸 빈 문자열 `""`로 바꾸니 `-`, `.`, 공백, `+`가 모두 사라지고 **숫자만** 남았습니다. 형식이 제각각이던 전화번호가 드디어 통일됐어요. 단순 `split`으로는 불가능했던 일입니다.

> **읽는 법:** `product_name`에는 앞뒤 공백, 이중 공백 등 오염이 있었지만, 정규식은 **위치와 상관없이** `P-숫자` 패턴만 정확히 찾아냅니다. `()`로 묶은 부분이 `extract`의 결과가 돼요. 이렇게 뽑은 깔끔한 코드로는 상품 테이블과 연결(merge, D+004)하기도 쉬워집니다.

> **읽는 법:** `findall`은 일치하는 **모든** 조각을 리스트로 돌려줍니다. `[A-Z]+`(대문자 묶음)로 보면 `summer25`는 대문자가 없어 `[]`(빈 리스트)가 나오죠 — 소문자로 쓴 쿠폰이라는 신호입니다. `\d+`(숫자 묶음)는 `VIP-2025`에서 `2025`만 깔끔히 뽑습니다.

## 단순 `str`로 충분할 때 vs 정규식이 필요할 때

정규식은 강력하지만 **읽기 어렵습니다.** 그래서 "꼭 필요할 때만" 씁니다. 판단 기준은 이렇습니다.

| 상황 | 추천 도구 |
| --- | --- |
| 구분자/위치가 **일정**하다 (`@`, `,`, 고정 자리) | `.str.split()`, 슬라이싱 (Part 2) |
| 단순히 "포함하는가" | `.str.contains("단어")` |
| **자리가 제각각**인 패턴 (전화번호, 코드, 날짜 조각) | 정규식 `extract`/`replace` |
| "숫자만", "특정 형식만" 뽑기 | 정규식 |

> 💡 **핵심:** "이건 단순 `split`으로 되나?"를 먼저 묻고, 안 될 때 정규식을 꺼내세요. 같은 결과라면 더 읽기 쉬운 쪽이 좋은 코드입니다.

> 📌 **다른 산업에서는?** 정규식은 패턴이 있는 모든 텍스트에 쓰입니다. 금융은 계좌·카드번호 형식 검증에, 헬스케어는 진단 코드(예: `A01.1`) 추출에, IT 보안은 로그에서 IP·URL을 뽑아내는 데 정규식을 씁니다(오늘 종합 실습에서 직접 해봅니다!).

### 스스로 해보자! ✏️ (3)

> 정답은 하나가 아닙니다. 일단 실행해보세요. 처음 보는 기호가 많아 낯선 게 당연합니다. 천천히요.

1. **(따라하기)** `orders["coupon"]`에서 `\d+`(숫자 묶음)를 `extract`해 `coupon_num` 컬럼을 만들어보세요.
2. **(응용)** `customers["phone_digits"]`의 글자 수(`.str.len()`)를 세어, 11자리가 아닌(국제표기 등) 번호가 있는지 확인하세요.
3. **(생각해보기)** `\d+`와 `[^0-9]`는 어떻게 반대 개념인가요? 한 문장으로 설명해보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
orders["coupon_num"] = orders["coupon"].str.extract(r"(\d+)")
print(orders[["coupon", "coupon_num"]].head(8))   # 쿠폰 없는 행은 NaN

print(customers["phone_digits"].str.len().value_counts())
# '+82 10 ...' 표기는 앞의 82가 붙어 자릿수가 달라질 수 있습니다.
```

3번: `\d`는 "숫자인 글자", `[^0-9]`는 "숫자가 **아닌** 글자"로 정확히 반대입니다. 그래서 "숫자만 뽑기"는 `\d+`를 추출하거나, `[^0-9]`를 삭제하는 두 방향 모두로 풀 수 있어요.

</details>

### ✅ 짚고 넘어가기

1. `\d`, `+`, `()`는 정규식에서 각각 무슨 뜻인가요?
2. `extract`와 `findall`의 차이는 무엇인가요?
3. 단순 `.str.split()`으로 충분한 상황과 정규식이 필요한 상황을 하나씩 들 수 있나요?

> 💡 **다음 Part 예고:** 지금까지는 '글자'를 다뤘습니다. 이제 데이터에서 가장 까다로운 글자, **날짜**로 갑니다. `signup_date`·`order_datetime`은 아직 *문자열*이라 계산이 안 돼요. 다음 Part에서 이 문자열을 진짜 날짜(datetime)로 바꿉니다.

In [34]:
# 예제: replace로 전화번호에서 숫자만 남기기 (숫자가 아닌 모든 글자 삭제)
# [^0-9] = '숫자가 아닌' 글자, 이것을 빈 문자열로 치환 → 숫자만 남음
customers["phone_digits"] = customers["phone"].str.replace(r"[^0-9]", "", regex=True)

print("[정제 전 → 후]")
for before, after in zip(customers["phone"].head(4), customers["phone_digits"].head(4)):
    print(f"{before:<18} →  {after}")

[정제 전 → 후]
010-8973-9577      →  01089739577
01090709666        →  01090709666
010.9609.7490      →  01096097490
+82 10 4600 7783   →  821046007783


In [35]:
# 예제: extract로 상품명에서 상품 코드(P-숫자)만 뽑기
# 패턴 (P-\d+) = 'P-' 다음에 숫자가 1개 이상 → 괄호로 묶어 그 부분만 추출
orders["product_code"] = orders["product_name"].str.extract(r"(P-\d+)")

print(orders[["product_name", "product_code"]].head(6))

            product_name product_code
0      무선 이어폰 (P-1001)         P-1001
1     USB-C 충전기 (P-1005)       P-1005
2       기계식 키보드 (P-1004)       P-1004
3    블루투스 스피커 (P-1002)         P-1002
4        무선 이어폰 (P-1001)       P-1001
5         보조배터리 (P-1006)       P-1006


In [36]:
# 예제: findall로 쿠폰 코드의 '영문 대문자 묶음'과 '숫자 묶음' 분리해 보기
sample = pd.Series(["SALE2025", "WELCOME10", "VIP-2025", "summer25", ""])

print("대문자 알파벳 덩어리 찾기  [A-Z]+ :")
print(sample.str.findall(r"[A-Z]+").tolist())
print("\n숫자 덩어리 찾기  \\d+ :")
print(sample.str.findall(r"\d+").tolist())

대문자 알파벳 덩어리 찾기  [A-Z]+ :
[['SALE'], ['WELCOME'], ['VIP'], [], []]

숫자 덩어리 찾기  \d+ :
[['2025'], ['10'], ['2025'], ['25'], []]


In [40]:
# 스스로 해보자! (3)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

# 1) 쿠폰에서 숫자만 추출
orders["coupon_num"] = orders["coupon"].str.extract(r"(\d+)")   # 힌트: (\d+)
print(orders[["coupon", "coupon_num"]].head(8))

# 2) 전화번호 자릿수 분포
print("\n",customers["phone_digits"].str.len().value_counts())     # 힌트: len

      coupon coupon_num
0   FREESHIP        NaN
1  WELCOME10         10
2                   NaN
3                   NaN
4                   NaN
5  WELCOME10         10
6                   NaN
7   SALE2025       2025

 phone_digits
11    150
12     50
Name: count, dtype: int64


## 날짜 파싱 — 문자열을 진짜 날짜로

# 4. 날짜 파싱 — 문자열을 진짜 날짜로

`order_datetime`을 떠올려봅시다. 화면에는 `2025-01-12 10:15:32`처럼 날짜로 보이지만, 컴퓨터에게는 그냥 **글자 덩어리(문자열)** 입니다. 그래서 "두 주문의 시간 차이", "주문이 많은 요일"을 계산할 수 없어요. 날짜로 계산하려면 먼저 **진짜 날짜 자료형(datetime)으로 변환(파싱)** 해야 합니다.

> ❓ **이 파트에서 답할 질문:** 포맷이 뒤섞이고 오류값까지 있는 날짜 문자열을, 어떻게 안전하게 진짜 날짜로 바꿀까요?

## 💡 쉽게 말하면 — '날짜처럼 보이는 글자' ≠ '날짜'

```text
문자열 "2025-01-12"   →  컴퓨터: "그냥 글자 10개"     (뺄셈·요일 계산 불가)
                          ↓  파싱(parsing)
datetime 2025-01-12  →  컴퓨터: "2025년 1월 12일"   (날짜 계산 가능!)
```

파싱(parsing)이란 **"이 글자를 이런 의미의 날짜로 해석하라"** 고 알려주는 일입니다. pandas의 `pd.to_datetime()`이 그 역할을 합니다.

## 자세히 알아보기 — `pd.to_datetime`의 세 가지 무기

| 옵션 | 하는 일 | 언제 |
| --- | --- | --- |
| `pd.to_datetime(s)` | 알아서 형식을 추론해 변환 | 형식이 깔끔할 때 |
| `format="..."` | 형식을 직접 지정 (가장 정확·빠름) | 형식을 알 때 |
| `format="mixed"` | 행마다 형식이 다를 때 각자 추론 | 포맷이 섞였을 때 |
| `errors="coerce"` | 변환 못 하는 값은 `NaT`(결측 날짜)로 | 오류값이 섞였을 때 |

형식 지정에 쓰는 기호도 함께 봅니다.

| 기호 | 뜻 | 예 |
| --- | --- | --- |
| `%Y` | 4자리 연도 | `2025` |
| `%m` | 2자리 월 | `01` |
| `%d` | 2자리 일 | `12` |
| `%H:%M:%S` | 시:분:초 | `10:15:32` |

> ⚠ **주의:** `NaT`(Not a Time)는 날짜판 결측치입니다. 숫자의 `NaN`과 같은 역할이에요. `errors="coerce"`를 쓰면 깨진 날짜가 오류를 내며 멈추는 대신 조용히 `NaT`가 됩니다.

> **읽는 법:** `dtype`이 `object`면 문자열이라는 뜻입니다. `2025-01-12`와 `2025/01/12`가 섞여 있죠. 게다가 Part 0에서 `처리중`, `-`, `unknown` 같은 오류값도 일부러 심어 두었습니다. 이걸 그냥 `pd.to_datetime`에 넣으면 오류값에서 멈춰버립니다 — 그래서 `errors="coerce"`가 필요해요.

> **읽는 법:** 이제 `dtype`이 `datetime64[ns]`로 바뀌었습니다 — 진짜 날짜가 됐다는 뜻이에요. 오류값(`처리중` 등)은 멈춤 대신 `NaT`로 처리돼, 변환 실패 건수까지 셀 수 있습니다. **"코드가 멈추지 않게 하면서, 문제 건수는 파악하기"** — 이것이 `coerce`의 가치입니다.

> **읽는 법:** 날짜 파싱은 종종 **문자열 정제(Part 1~3)와 한 몸**입니다. 여기서는 정규식으로 구분자를 모두 없애 `20240315` 형태로 통일한 뒤, `format="%Y%m%d"`로 정확히 파싱했어요. 형식을 명시하면 추론보다 빠르고 안전합니다. 오늘 배운 도구들이 이렇게 이어집니다.

## 데이터로 확인해 봅시다

- 날짜 파싱은 **모든 시계열 분석의 출입문**입니다. 파싱이 안 된 채로는 추세도, 요일 패턴도, 기간 필터도 불가능해요.
- 실무에서는 `errors="coerce"`로 **깨진 날짜를 `NaT`로 모은 뒤 그 개수를 품질 지표로 보고**합니다. "전체 2,000건 중 12건 파싱 실패(0.6%)"처럼요.

> 📌 **다른 산업에서는?** 날짜 파싱은 어디서나 첫 단추입니다. 금융은 거래 타임스탬프를 파싱해 일별 잔고를 만들고, 제조는 설비 로그 시각을 파싱해 고장 간격을 재며, 헬스케어는 검사일을 파싱해 추적 기간을 계산합니다.

### 스스로 해보자! ✏️ (4)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `orders["order_dt"]`에서 **가장 이른 주문일**과 **가장 늦은 주문일**을 출력하세요. (힌트: `.min()`, `.max()`)
2. **(응용)** `order_dt`가 `NaT`인(파싱 실패한) 주문만 골라, 원본 `order_datetime`에 어떤 값이 있었는지 확인하세요.
3. **(생각해보기)** `errors="coerce"` 대신 아무 옵션도 안 줬다면 어떤 일이 벌어졌을까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
print("가장 이른 주문:", orders["order_dt"].min())
print("가장 늦은 주문:", orders["order_dt"].max())

failed = orders[orders["order_dt"].isna()]
print(failed["order_datetime"].value_counts())   # '처리중', '-', 'unknown'
```

3번: 옵션이 없으면 `처리중` 같은 값을 만나는 순간 `pd.to_datetime`이 **오류를 내며 멈춥니다**. 데이터가 많을수록 "어디서 멈췄는지" 찾기 어렵죠. `coerce`는 멈추지 않고 문제값을 `NaT`로 모아주어 나중에 한꺼번에 점검하게 해줍니다.

</details>

### ✅ 짚고 넘어가기

1. 문자열 `"2025-01-12"`와 datetime `2025-01-12`는 컴퓨터 입장에서 무엇이 다른가요?
2. `format="mixed"`와 `errors="coerce"`는 각각 어떤 상황을 위한 옵션인가요?
3. `NaT`는 무엇이고, 숫자의 어떤 값에 해당하나요?

> 💡 **다음 Part 예고:** 이제 진짜 날짜가 생겼습니다. 그런데 날짜 하나에는 연·월·일·요일·시각이 **다 들어 있죠.** 다음 Part에서는 이 날짜에서 "요일"이나 "시각" 같은 부품만 꺼내, 분석용 컬럼으로 만드는 법을 배웁니다.

In [13]:
# 예제: 변환 전 — order_datetime은 그냥 문자열(object)
print("dtype:", orders["order_datetime"].dtype)   # object = 문자열
print(orders["order_datetime"].head(6).tolist())

# 이 상태에서는 '시간 차이' 같은 계산이 불가능합니다.
print("\n'문자열'이라 정렬해도 날짜 순서가 보장되지 않습니다.")

dtype: str
['2025-01-01 23:12:56', '2025/01/25 12:15:09', '2025-01-31 14:14:25', '2025/03/30 02:12:54', '2025-02-18 23:06:24', '2025/02/18 07:41:05']

'문자열'이라 정렬해도 날짜 순서가 보장되지 않습니다.


In [14]:
# 예제: format='mixed' + errors='coerce'로 안전하게 변환
orders["order_dt"] = pd.to_datetime(
    orders["order_datetime"],
    format="mixed",        # 행마다 다른 형식을 각자 추론
    errors="coerce",       # 변환 못 하면 NaT(결측 날짜)
)

print("변환 후 dtype:", orders["order_dt"].dtype)   # datetime64[ns]
print("변환 실패(NaT) 건수:", orders["order_dt"].isna().sum())   # 오류값 12건
print(orders[["order_datetime", "order_dt"]].head(6))

변환 후 dtype: datetime64[us]
변환 실패(NaT) 건수: 12
        order_datetime            order_dt
0  2025-01-01 23:12:56 2025-01-01 23:12:56
1  2025/01/25 12:15:09 2025-01-25 12:15:09
2  2025-01-31 14:14:25 2025-01-31 14:14:25
3  2025/03/30 02:12:54 2025-03-30 02:12:54
4  2025-02-18 23:06:24 2025-02-18 23:06:24
5  2025/02/18 07:41:05 2025-02-18 07:41:05


In [15]:
# 예제: 구분자 통일 후 format 지정하기 (정제 → 파싱 연결)
# signup_date는 2024-03-15 / 2024/03/15 / 2024.03.15 / 20240315 가 섞여 있음
# 1단계: 정규식으로 구분자(- / .)를 모두 제거 → 'YYYYMMDD' 8자리로 통일 (Part 1~3 복습!)
digits_only = customers["signup_date"].str.replace(r"[^0-9]", "", regex=True)

# 2단계: 형식을 명시해 파싱 (가장 정확)
customers["signup_dt"] = pd.to_datetime(digits_only, format="%Y%m%d", errors="coerce")

print(customers[["signup_date", "signup_dt"]].head(6))
print("\n변환 실패 건수:", customers["signup_dt"].isna().sum())

  signup_date  signup_dt
0  2024-04-27 2024-04-27
1  2024/09/26 2024-09-26
2  2024.03.20 2024-03-20
3    20240402 2024-04-02
4  2024-09-22 2024-09-22
5  2024/11/27 2024-11-27

변환 실패 건수: 0


In [ ]:
# 스스로 해보자! (4)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

# 1) 기간 확인
# print("가장 이른 주문:", orders["order_dt"].____())   # min
# print("가장 늦은 주문:", orders["order_dt"].____())   # max

# 2) 파싱 실패 행의 원본 보기
# failed = orders[orders["order_dt"].____()]            # isna
# print(failed["order_datetime"].value_counts())

## `dt` 기능 — 날짜에서 부품 꺼내기

# 5. `dt` 기능 — 날짜에서 부품 꺼내기

Part 4에서 `order_dt`라는 진짜 날짜를 얻었습니다. 그런데 동료의 원래 요청은 "**요일별·시간대별** 매출"이었죠. 날짜 `2025-01-12 10:15:32` 안에는 연도·월·일·요일·시각이 모두 들어 있습니다. 이 중 필요한 **부품만 꺼내** 분석용 컬럼으로 만들어야 합니다.

> ❓ **이 파트에서 답할 질문:** 하나의 날짜 값에서 연·월·요일·시각 같은 부품을 어떻게 꺼낼까요?

## 💡 쉽게 말하면 — 날짜는 '부품이 든 상자'

```text
2025-01-12 10:15:32
└─┬─┘ └┬┘└┬┘ └──┬──┘
 연도  월  일   시각        ← 한 상자 안에 여러 부품
        │
   .dt.year   → 2025
   .dt.month  → 1
   .dt.day    → 12
   .dt.hour   → 10
   .dt.dayofweek → 6 (일요일)   ※ 0=월, 1=화, ... , 6=일
```

문자열에 `.str`를 붙였듯, **날짜에는 `.dt`** 를 붙여 부품을 꺼냅니다. 짝을 이루는 개념이에요.

## 자세히 알아보기 — 자주 쓰는 `.dt` 부품

| 코드 | 꺼내는 부품 | 결과 예 |
| --- | --- | --- |
| `.dt.year` | 연도 | `2025` |
| `.dt.month` | 월 (1~12) | `1` |
| `.dt.day` | 일 | `12` |
| `.dt.hour` | 시각(시) | `10` |
| `.dt.dayofweek` | 요일 번호 (0=월 … 6=일) | `6` |
| `.dt.day_name()` | 요일 이름(영문) | `"Sunday"` |
| `.dt.date` | 시각을 뺀 날짜 부분 | `2025-01-12` |

> 💡 **개념 연결:** `.dt`로 만든 부품 컬럼은 보통 숫자나 범주값입니다. 그래서 D+004에서 배운 `groupby`에 바로 넣어 "요일별 평균 매출", "시간대별 주문 수"를 구할 수 있어요. 오늘 만드는 부품이 곧 집계의 재료가 됩니다.

> **읽는 법:** 날짜 하나에서 여러 부품이 깔끔히 분리됐습니다. `dayofweek`는 숫자(0=월요일), `day_name()`은 영문 이름을 줍니다. 그래프 라벨로는 한글 폰트 문제를 피하려 영문 이름이 편하고, 정렬·집계에는 숫자가 편해요. 그래서 둘 다 만들어 두는 경우가 많습니다.

> **읽는 법:** 막대 길이로 요일별 평균 주문 금액을 비교합니다. `.dt`로 꺼낸 요일 부품과 D+004의 `groupby`가 만나 "어느 요일에 큰 주문이 많은가"라는 비즈니스 질문에 답했어요. 날짜 정제가 끝나니 비로소 인사이트가 나옵니다.

## 데이터로 확인해 봅시다

- `.dt.hour`로 **시간대별 트래픽**, `.dt.dayofweek`로 **요일 패턴**, `.dt.month`로 **계절성**을 봅니다. 모두 "언제 무슨 일이 일어나는가"를 푸는 핵심 부품이에요.
- 주말 여부(`dayofweek >= 5`), 영업시간 여부(`9 <= hour < 18`) 같은 **파생 플래그**도 부품에서 한 줄로 만듭니다.

> 📌 **다른 산업에서는?** 시간 부품은 분야마다 다른 질문에 답합니다. 콜센터는 `.dt.hour`로 **시간대별 콜 수**를 보고 인력을 배치하고, 제조는 `.dt.dayofweek`로 **요일별 불량률**을 점검하며, 헬스케어는 `.dt.month`로 **계절성 질환 추이**를 분석합니다.

### 스스로 해보자! ✏️ (5)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

`valid` 데이터(파싱 성공한 주문)를 사용합니다.

1. **(따라하기)** `hour`(시각) 부품으로 **시간대별 주문 건수**를 세어보세요. (힌트: `valid["hour"].value_counts().sort_index()`)
2. **(응용)** `dow_num >= 5`(토·일)인 주문만 골라 **주말 주문 비율**을 구해보세요.
3. **(생각해보기)** "심야(0~6시) 주문 비중"을 알고 싶다면 어떤 부품과 조건을 쓰면 될까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
print(valid["hour"].value_counts().sort_index())

weekend = valid[valid["dow_num"] >= 5]
print("주말 주문 비율:", round(len(weekend) / len(valid) * 100, 1), "%")

# 3) 심야 주문: hour 부품에 범위 조건
night = valid[(valid["hour"] >= 0) & (valid["hour"] < 6)]
print("심야 주문 비율:", round(len(night) / len(valid) * 100, 1), "%")
```

`.dt`로 꺼낸 부품은 그냥 숫자 컬럼이라, D+003의 불리언 필터링·D+004의 집계가 모두 그대로 통합니다.

</details>

### ✅ 짚고 넘어가기

1. 문자열의 `.str`에 대응하는, 날짜의 접근자는 무엇인가요?
2. `.dt.dayofweek`에서 `0`과 `5`는 각각 무슨 요일인가요?
3. `.dt`로 꺼낸 부품을 `groupby`에 넣으면 무엇을 할 수 있나요?

> 💡 **다음 Part 예고:** 부품으로 "요일별"은 봤습니다. 그런데 "**일별·주별 매출 추세**"는요? 날짜를 일/주/월 단위로 묶어 집계하는 일은 너무 자주 해서, pandas에 전용 도구가 있습니다. 다음 Part의 **리샘플링**이에요.

In [ ]:
# 예제: 날짜에서 부품 꺼내 새 컬럼 만들기
valid = orders.dropna(subset=["order_dt"]).copy()   # NaT(파싱 실패) 제외

valid["year"] = valid["order_dt"].dt.year
valid["month"] = valid["order_dt"].dt.month
valid["hour"] = valid["order_dt"].dt.hour
valid["dow_num"] = valid["order_dt"].dt.dayofweek      # 0=월 ... 6=일
valid["dow_name"] = valid["order_dt"].dt.day_name()    # 영문 요일 이름

print(valid[["order_dt", "year", "month", "hour", "dow_num", "dow_name"]].head())

In [ ]:
# 예제: 부품 + groupby = '요일별 평균 매출' (D+004 복습과 결합)
dow_sales = valid.groupby("dow_name")["amount"].mean()

# 요일 순서를 월→일로 정렬 (영문 이름 순서 지정)
order_days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_sales = dow_sales.reindex(order_days)

print("[요일별 평균 주문 금액]")
print(dow_sales.round(0))

In [ ]:
# [도식] 요일별 평균 매출 막대그래프 — 부품이 곧 인사이트가 된다
plt.figure(figsize=(9, 4))
sns.barplot(x=dow_sales.index, y=dow_sales.values, color="#4C72B0")
plt.title("Average Order Amount by Day of Week")
plt.xlabel("day of week"); plt.ylabel("avg amount")
plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# 스스로 해보자! (5)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

# 1) 시간대별 주문 건수
# print(valid["hour"].____().sort_index())              # value_counts

# 2) 주말 주문 비율
# weekend = valid[valid["dow_num"] ___ 5]               # >= 5 (토=5, 일=6)
# print("주말 주문 비율:", round(len(weekend) / len(valid) * 100, 1), "%")

## 리샘플링·이동평균 — 시간 단위로 묶어 추세 읽기

# 6. 리샘플링·이동평균 — 시간 단위로 묶어 추세 읽기

Part 5에서 요일별 평균을 봤습니다. 이번엔 "**날짜 흐름에 따른 매출 추세**"가 궁금합니다. 90일치 주문을 그대로 점으로 찍으면 들쭉날쭉해서 추세가 안 보여요. 그래서 **하루(또는 한 주) 단위로 묶어** 합계를 내고, 들쭉날쭉함을 부드럽게 펴 줄 도구가 필요합니다.

> ❓ **이 파트에서 답할 질문:** 시계열 데이터를 분석 목적에 맞는 시간 단위(일·주·월)로 어떻게 묶고, 추세를 어떻게 부드럽게 볼까요?

## 💡 쉽게 말하면 — '시간 자(ruler)'를 바꿔 끼우기

```text
원본 주문(초 단위, 들쭉날쭉)
 ●● ● ●●● ●  ● ●● ●●●● ...

   ↓ resample("D"): 하루 단위로 묶어 합계
 [1/1] [1/2] [1/3] [1/4] ...   ← 점이 줄어 추세가 보임

   ↓ rolling(7): 최근 7일 평균으로 부드럽게
 ～～～～～～～                  ← 잔물결을 펴 큰 흐름만
```

- **리샘플링(resampling)**: 시간 축의 '눈금 간격'을 바꾸는 일. 초 단위 → 일 단위로 묶어 합계·평균을 냅니다.
- **이동평균(rolling/moving average)**: 매 시점에서 "최근 N개"의 평균을 내, 들쭉날쭉함을 부드럽게 펴는 일.

## 자세히 알아보기 — `resample`과 `rolling`

`resample`은 **날짜가 인덱스(index)** 여야 동작합니다. 그래서 `set_index("order_dt")`로 날짜를 인덱스로 올린 뒤 씁니다.

| 코드 | 하는 일 |
| --- | --- |
| `df.set_index("order_dt")` | 날짜를 인덱스(시간 축)로 |
| `.resample("D").sum()` | 하루 단위로 묶어 합계 (`"D"`=일) |
| `.resample("W").sum()` | 한 주 단위로 묶어 합계 (`"W"`=주) |
| `.resample("h").count()` | 한 시간 단위로 묶어 개수 (`"h"`=시간) |
| `.rolling(7).mean()` | 최근 7개의 이동평균 |

| 빈도 문자 | 의미 |
| --- | --- |
| `"h"` | 시간(hour) |
| `"D"` | 일(day) |
| `"W"` | 주(week) |
| `"ME"` | 월말(month end) |

> 💡 **핵심:** 분석 목적이 시간 단위를 결정합니다. "시간대별 트래픽"이면 `"h"`, "일 매출 추세"면 `"D"`, "월별 KPI"(D+004)면 `"ME"`. 먼저 *무엇을 보고 싶은지* 정한 뒤 자를 고르세요.

> **읽는 법:** 초 단위로 흩어졌던 2,000건의 주문이 하루 단위 합계로 묶였습니다. 이제 행 하나가 "그 날의 총매출"이에요. `resample`은 날짜 인덱스가 있어야 동작한다는 점, 그래서 `set_index`를 먼저 한 점을 기억하세요.

> **읽는 법:** 옅은 선(일별)은 위아래로 튀지만, 굵은 빨간 선(7일 이동평균)은 **큰 흐름**만 부드럽게 보여줍니다. 이동평균의 앞 6일은 "최근 7일"을 채울 데이터가 없어 비어 있어요(`NaN`). 추세를 설명할 때는 이동평균, 특정 날짜의 값이 궁금하면 원본 — 목적에 따라 골라 봅니다.

> **읽는 법:** 같은 데이터인데 자를 `"D"`에서 `"W"`로 바꾸니 행이 더 줄고 흐름이 더 단순해졌습니다. 너무 잘게 자르면(`"h"`) 노이즈가 많고, 너무 크게 자르면(`"ME"`) 디테일이 사라져요. **목적에 맞는 자 고르기**가 시계열 분석의 감각입니다.

## 데이터로 확인해 봅시다

- `resample`은 **대시보드의 기본 재료**입니다. 일/주/월 매출 그래프는 거의 다 리샘플링으로 만듭니다.
- `rolling`(이동평균)은 **추세 판단**에 씁니다. 주가의 이동평균선, 코로나 확진자 7일 평균이 같은 원리예요.

> 📌 **다른 산업에서는?** 시간 단위로 묶기는 어디서나 핵심입니다. 금융은 분 단위 체결을 일 단위로 리샘플링해 캔들 차트를 만들고, IT 운영은 초 단위 로그를 시간 단위로 묶어 트래픽을 보며, 제조는 이동평균으로 센서 값의 추세 이탈을 감지합니다.

### 스스로 해보자! ✏️ (6)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

`ts`(날짜를 인덱스로 올린 데이터)를 사용합니다.

1. **(따라하기)** **시간대별 주문 건수**를 보려고 합니다. `resample("h")`로 묶어 개수(`.size()` 또는 `["amount"].count()`)를 구해보세요.
2. **(응용)** 일별 매출의 **3일 이동평균**(`rolling(3)`)을 구해 7일 이동평균과 비교해보세요. 어느 쪽이 더 부드러운가요?
3. **(생각해보기)** "월별 매출"을 보고 싶다면 빈도 문자로 무엇을 써야 할까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
hourly = ts["amount"].resample("h").count()
print(hourly.head(24))

ma3 = daily_sales.rolling(3).mean()
print(pd.DataFrame({"daily": daily_sales, "ma3": ma3, "ma7": daily_ma7}).head(10))
# 창(window)이 클수록(7 > 3) 더 부드럽지만, 그만큼 최근 변화에 둔해집니다.
```

3번: 월별은 `"ME"`(month end) 또는 `"MS"`(month start)를 씁니다. `ts["amount"].resample("ME").sum()`.

</details>

### ✅ 짚고 넘어가기

1. `resample`을 쓰기 전에 반드시 해야 하는 준비는 무엇인가요? (힌트: 인덱스)
2. `"D"`, `"W"`, `"h"`는 각각 어떤 시간 단위인가요?
3. 이동평균(`rolling`)은 무엇을 위해 쓰며, 창(window)을 키우면 그래프가 어떻게 변하나요?

> 💡 **다음 단계 예고:** 이제 문자열·정규식·날짜 도구를 모두 익혔습니다. 마지막으로, 이 모든 걸 **한 번에** 써서 가장 지저분한 데이터 — 한 줄짜리 **웹 로그** — 를 표로 바꾸는 종합 실습으로 넘어갑니다.

# 🧪 종합 실습 — 웹 로그를 집계 테이블로 만들기

지금까지 배운 **정규표현식 + 날짜 파싱 + dt 부품 + 리샘플링**을 한 흐름에서 모두 써봅니다.

모두마켓 운영팀이 서버 접속 로그(web_logs)를 건넸습니다. 그런데 한 줄이 통째로 **하나의 문자열**입니다. 흔히 보는 웹 서버 로그 형식이에요.

```text
203.0.113.5 - - [12/Jan/2025:10:15:32] "GET /products/123 HTTP/1.1" 200 1024
└────┬────┘        └───────┬────────┘  └┬┘ └─────┬─────┘        └┬┘ └┬┘
   IP 주소              접속 시각      방식   요청 경로          상태  크기
```

이 한 줄에서 **IP·시각·요청 방식·경로·상태 코드·응답 크기**를 뽑아내, 분석 가능한 표로 만드는 것이 목표입니다. 아래 셀을 실행해 로그를 생성한 뒤, 세 개의 시나리오를 차례로 해결하고 마지막에 **트래픽 리포트**를 작성하세요.

> 💡 순서가 중요합니다. **원본 관찰 → 정규식 파싱 → 날짜 변환 → 부품/집계** 순으로 진행합니다. (오늘 Part 1~6의 순서 그대로예요!)

## 시나리오 1 — 원본 관찰 + 정규식 파싱

가장 먼저 **원본을 눈으로 봅니다**(오늘의 핵심 태도!). 위 출력에서 IP·시각·방식·경로·상태·크기가 각각 어디에 있는지 확인하세요. 그런 다음 **정규표현식(Part 3)** 으로 여섯 조각을 한 번에 추출합니다.

`.str.extract()`에 **이름이 붙은 그룹** `(?P<이름>패턴)`을 쓰면, 각 그룹이 그 이름의 컬럼이 되어 한 번에 표가 만들어집니다.

> **읽는 법:** 한 줄짜리 문자열이 6개 컬럼의 표로 바뀌었습니다. 이름 붙은 그룹 `(?P<ip>...)` 덕분에 컬럼명까지 자동으로 붙었어요. 추출 실패가 0건이면 모든 줄이 패턴에 맞았다는 뜻입니다. 정규식 하나가 `split` 여러 번을 대신했습니다.

## 시나리오 2 — 날짜 변환 + 타입 정리 + dt 부품

추출한 `ts`는 아직 **문자열**(`12/Jan/2025:10:15:32`)이고, `status`·`size`도 문자열입니다. **날짜 파싱(Part 4)** 으로 `ts`를 진짜 날짜로 바꾸고, 숫자 컬럼은 숫자로 변환한 뒤, **dt 부품(Part 5)** 으로 시각(hour)을 꺼냅니다.

`%d/%b/%Y:%H:%M:%S` 형식을 직접 지정합니다. (`%b`는 `Jan`·`Feb` 같은 **영문 월 약자**를 뜻해요.)

> **읽는 법:** 이제 `ts`는 `datetime64`, `status`·`size`는 정수가 됐습니다. 각 컬럼이 제 역할의 자료형을 갖춰야 비로소 계산·집계가 가능해요. Part 4(파싱)와 Part 5(부품)가 한 셀에서 이어졌습니다.

> **읽는 법:** `status >= 400`은 오류 응답(404 없음, 500 서버오류 등)입니다. 불리언의 `.mean()`은 "참의 비율"이라 곧 오류율이 됩니다(D+003 복습). 인기 경로는 `value_counts()`로 한 줄이죠. 파싱이 끝나니 이런 질문에 즉시 답할 수 있습니다.

## 시나리오 3 — 리샘플링으로 시간대별 트래픽

마지막으로 **리샘플링(Part 6)** 입니다. `ts`를 인덱스로 올리고 **시간 단위(`"h"`)** 로 묶어, 시간대별 요청 수(트래픽)를 집계하고 그래프로 봅니다.

> **읽는 법:** 초 단위 로그 1,500줄이 시간당 요청 수로 묶여, 트래픽이 언제 몰리는지 한눈에 보입니다. IP·시각·경로가 다 문자열로 엉켜 있던 원본이, 오늘 배운 도구들의 연쇄(정규식→파싱→부품→리샘플링)를 거쳐 **운영 의사결정에 쓰는 표와 그래프**가 됐어요.

## 📊 웹 트래픽 리포트 작성 (제출물)

지금까지의 결과를 아래 **리포트 양식**에 맞춰 마크다운으로 정리하세요. 이것이 오늘의 최종 산출물이며, 과제로 제출합니다.

```markdown
# 모두마켓 웹 트래픽 — 로그 분석 리포트

## 1. 데이터 개요
- 로그 줄 수: ___줄
- 기간: 2025-03-01 ~ ___
- 추출 필드: ip, ts, method, path, status, size

## 2. 파싱 (어떻게 표로 만들었나)
- 사용한 정규식 패턴의 핵심: IP는 `\d+\.\d+...`, 시각은 `[^\]]+` 로 추출
- 날짜 형식: %d/%b/%Y:%H:%M:%S 로 파싱, 실패 ___건

## 3. 트래픽 요약
- 총 요청 수: ___
- 오류 응답(4xx·5xx) 비율: ___%
- 요청 많은 경로 TOP 3: ___

## 4. 시간 패턴 (리샘플링에서 읽은 것)
- 트래픽이 가장 몰린 시간대: ___시
- 요일/시간대 특징: ___

## 5. 다음 분석 제안
- (오류가 잦은 경로 점검, 특정 IP의 비정상 요청 탐지 등)
```

> 💡 빈칸(___)은 위 셀들의 실행 결과를 보고 직접 채워 넣으세요. **"왜 그렇게 판단했는지"** 한 줄씩 근거를 적으면 훌륭한 리포트가 됩니다.

# ✅ 오늘의 퀴즈

배운 내용을 잠깐 확인해볼게요. 틀려도 괜찮습니다. "이런 걸 배웠지" 하고 떠올리는 용도예요.

### 개념 퀴즈

1. 문자열 컬럼의 앞뒤 공백을 없애고 소문자로 통일하려면? `(a) .str.split()  (b) .str.strip().str.lower()  (c) .dt.lower()`
2. "자리는 제각각이지만 규칙(숫자 등)은 있는" 패턴을 뽑을 때 쓰는 도구는 무엇인가요?
3. 날짜 문자열에 오류값이 섞여 있을 때, 변환을 멈추지 않고 `NaT`로 처리하는 옵션은? (힌트: `errors=...`)
4. 초 단위 데이터를 "하루 단위 합계"로 묶으려면 어떤 메서드와 빈도 문자를 쓰나요?

### 코드 퀴즈

`customers`에서 **이메일 도메인이 `gmail.com`인 고객**의 수를 구하세요. (단, 원본 `email`에는 대소문자·공백 오염이 있으니 먼저 정제할 것)

> **읽는 법:** 정제(Part 1) → 분리·추출(Part 2)이 한 줄씩 이어졌습니다. 만약 정제를 건너뛰었다면 `GMAIL.COM`이나 `' gmail.com '`이 따로 세어져 **수가 틀렸을** 거예요. "정제가 집계의 전제"라는 오늘의 메시지가 여기에도 그대로 적용됩니다.

# 🎓 정리 & 다음 시간 예고

## 오늘 배운 것이 어떻게 이어졌나

```text
[Part 1] 문자열 정제(str)      공백·대소문자·표기 통일
   ↓  (통째로는 부족, 조각이 필요)
[Part 2] 검색·분리·추출         contains·split로 일부 골라내기
   ↓  (구분자·위치가 일정해야 함)
[Part 3] 정규표현식             자리 제각각인 패턴을 규칙으로 추출
   ↓  (가장 까다로운 텍스트 = 날짜)
[Part 4] 날짜 파싱(datetime)    문자열을 진짜 날짜로(coerce로 안전하게)
   ↓  (날짜 안의 부품 꺼내기)
[Part 5] dt 기능               연·월·요일·시각 → 분석 컬럼
   ↓  (시간 단위로 묶어 추세)
[Part 6] 리샘플링·이동평균       일·주 단위 집계 + 부드러운 추세선
   ↓
[종합 실습] 웹 로그 파싱        정규식→파싱→부품→리샘플링 전부 통합
```

## 한 장 정리표

| 주제 | 핵심 한 줄 | 대표 코드 |
| --- | --- | --- |
| 문자열 정제 | 공백·대소문자부터 맞춘다 | `.str.strip().str.lower()` |
| 검색·분리 | 일정한 구분자면 split | `.str.split("@").str[1]` |
| 정규표현식 | 제각각 패턴은 규칙으로 | `.str.extract(r"(\d+)")` |
| 날짜 파싱 | 문자열→날짜, 실패는 NaT | `pd.to_datetime(s, errors="coerce")` |
| dt 부품 | 날짜에서 부품 꺼내기 | `.dt.dayofweek`, `.dt.hour` |
| 리샘플링 | 목적에 맞는 시간 단위로 | `.resample("D").sum()` |

## 처리 신호 → 다음에 쓸 도구 (판단 지도)

오늘 데이터에서 만난 신호가 어떤 도구로 이어지는지 정리합니다.

| 데이터에서 본 신호 | 자연스러운 처리 도구 |
| --- | --- |
| `'서울'`/`'Seoul'`/`' 서울 '` 표기 혼재 | `.str.strip()` + `.str.replace()` |
| `@` 뒤 도메인만 필요 | `.str.split("@").str[1]` |
| 전화번호·코드처럼 자리가 제각각 | 정규식 `extract`/`replace` |
| `2025-01-12`가 문자열(object) | `pd.to_datetime(..., errors="coerce")` |
| "요일별·시간대별"이 궁금 | `.dt.dayofweek`, `.dt.hour` + `groupby` |
| "일·주 단위 추세"가 궁금 | `.resample("D"/"W")` + `.rolling()` |

## 🎓 다음 시간 예고

> **"수작업 정제를, 함수와 파이프라인으로 자동화한다."**
>
> 오늘 우리는 `.str.strip().str.lower()`, `pd.to_datetime(...)`을 **한 줄씩 손으로** 적었습니다. 그런데 새 데이터가 들어올 때마다 이 과정을 매번 다시 쓴다면? 다음 시간에는 이 정제 과정을 **함수로 묶고, `.pipe()`로 이어 붙여** 재사용 가능한 전처리 파이프라인으로 만드는 법을 배웁니다. 오늘의 "점으로 이어 붙이기(method chaining)"가 그 출발점이었어요.

# 📝 오늘의 과제

오늘 만든 **웹 트래픽 리포트**를 다듬어 GitHub에 제출합니다.

## 제출물

1. 종합 실습의 웹 로그를 파싱·집계한 노트북(`.ipynb`)
2. 마무리 양식을 채운 **웹 트래픽 리포트**(노트북 안 마크다운 셀로 작성)

## 필수 과제

- [ ] 정규식 `extract`로 로그에서 6개 필드(ip·ts·method·path·status·size)를 추출했다.
- [ ] `pd.to_datetime(..., errors="coerce")`로 시각을 파싱하고 실패 건수를 확인했다.
- [ ] `.dt`로 시각(hour) 부품을 꺼내고, `value_counts`로 상태 코드·인기 경로를 집계했다.
- [ ] `resample("h")`로 시간대별 트래픽을 집계하고 그래프로 그렸다.
- [ ] 마무리 리포트 양식(5개 항목)을 모두 채웠다.

## 심화 과제 (선택)

- [ ] 특정 IP의 요청 수를 세어, **비정상적으로 많은 요청을 보낸 IP**가 있는지 찾아보고 근거를 적었다.
- [ ] 오류 응답(4xx·5xx)이 가장 잦은 **경로(path)** 를 찾아, 운영팀에 무엇을 제안할지 적었다. (정답 없음 — 판단 근거가 핵심)

## 제출 방법 (GitHub)

개인 포트폴리오 저장소(공개)에 `D005/` 폴더를 만들어 노트북을 올립니다.

```bash
# 이번 주부터는 작업 브랜치에서 작업한 뒤 PR(Pull Request)로 제출해봅니다.
git checkout -b feat/d005-log-report
git add D005/
git commit -m "D005 웹 로그 파싱 및 트래픽 리포트"
git push origin feat/d005-log-report
# 이후 GitHub에서 main으로 Pull Request 생성 → 강사가 코드 리뷰
```

## 평가 기준

| 축 | 무엇을 보나 |
| --- | --- |
| 정확성 | 정규식·파싱·집계 코드가 오류 없이 돌고 수치가 맞는가 |
| 합리성 | 정제·파싱 전략이 데이터 형태에 맞게 선택됐는가 |
| 인사이트 | 트래픽·오류 패턴에서 "그래서 무엇을 할지"를 근거와 함께 적었는가 |

> 💡 모두의연구소의 과제는 순위를 매기지 않습니다. **"어제의 나보다 지저분한 데이터를 더 잘 길들이게 되었는가"** 가 기준이에요. 코드와 인사이트를 동료와 공유하며 함께 성장합니다.

---

수고하셨습니다! 🎉

오늘 여러분은 데이터에서 가장 손이 많이 가는 두 가지, **문자열과 날짜**를 길들였습니다. 사람 눈에만 읽히던 지저분한 텍스트를 컴퓨터가 집계할 수 있는 값으로 바꾸고, 글자 덩어리였던 날짜를 계산 가능한 시간으로 되살렸어요. 마지막엔 한 줄짜리 로그를 표와 그래프로 바꿔내기까지 했습니다.

다음 시간에는 이 모든 수작업을 **자동화하는 파이프라인**의 세계로 갑니다. 천천히, 그러나 멈추지 않고 가봅시다. 🚀

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 교안은 생성형 AI를 활용해 제작하고 제작자가 검수했습니다.<br>
무단 복제 및 배포를 금합니다.</sub>

In [ ]:
# 예제: 일별 매출 합계 (resample 'D')
ts = valid.set_index("order_dt").sort_index()   # 날짜를 인덱스로 + 시간순 정렬

daily_sales = ts["amount"].resample("D").sum()   # 하루 단위 합계
print("일별 매출 (앞 7일):")
print(daily_sales.head(7))
print("\n총 일수:", len(daily_sales))

In [ ]:
# 예제: 이동평균(rolling) — 들쭉날쭉한 일 매출을 7일 평균으로 부드럽게
daily_ma7 = daily_sales.rolling(7).mean()        # 최근 7일 이동평균

plt.figure(figsize=(11, 4))
plt.plot(daily_sales.index, daily_sales.values, alpha=0.4, label="Daily")
plt.plot(daily_ma7.index, daily_ma7.values, color="#C44E52", linewidth=2, label="7-day Moving Avg")
plt.title("Daily Sales and 7-day Moving Average")
plt.xlabel("date"); plt.ylabel("sales")
plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 예제: 주 단위로 자를 바꿔보기 (resample 'W')
weekly_sales = ts["amount"].resample("W").sum()
print("[주별 매출]")
print(weekly_sales)

In [ ]:
# 스스로 해보자! (6)
# 아래 주석(#)을 지우고 빈칸(___)을 채워보세요.

# 1) 시간대별 주문 건수
# hourly = ts["amount"].resample("___").count()    # 힌트: "h"
# print(hourly.head(24))

# 2) 3일 이동평균 vs 7일 이동평균
# ma3 = daily_sales.rolling(___).mean()            # 힌트: 3
# print(pd.DataFrame({"daily": daily_sales, "ma3": ma3, "ma7": daily_ma7}).head(10))

In [ ]:
# 웹 서버 접속 로그 생성 (가상) — 한 줄이 통째로 문자열
np.random.seed(7)
n_logs = 1500
ips = [f"203.0.113.{k}" for k in range(1, 60)] + [f"198.51.100.{k}" for k in range(1, 60)]
paths = ["/", "/products", "/products/123", "/products/456", "/cart",
         "/checkout", "/login", "/search", "/api/orders"]
methods = ["GET", "GET", "GET", "POST"]
statuses = [200, 200, 200, 200, 200, 404, 500, 301]

# 시각: 7일치, 초 단위로 무작위 발생 후 정렬
log_times = pd.to_datetime("2025-03-01") + pd.to_timedelta(
    np.sort(np.random.randint(0, 7 * 24 * 3600, n_logs)), unit="s")

lines = []
for i in range(n_logs):
    ip = np.random.choice(ips)
    ts_str = log_times[i].strftime("%d/%b/%Y:%H:%M:%S")   # 12/Jan/2025:10:15:32
    m = np.random.choice(methods)
    p = np.random.choice(paths)
    st = np.random.choice(statuses)
    sz = np.random.randint(200, 5000)
    lines.append(f'{ip} - - [{ts_str}] "{m} {p} HTTP/1.1" {st} {sz}')

web_logs = pd.DataFrame({"raw": lines})
print("웹 로그 생성 완료:", web_logs.shape)
print("\n[원본 로그 한 줄은 이렇게 생겼습니다]")
for line in web_logs["raw"].head(3):
    print(line)

In [ ]:
# 시나리오 1 — 정규식으로 6개 필드를 한 번에 추출
pattern = (
    r"(?P<ip>\d+\.\d+\.\d+\.\d+)"        # IP: 숫자.숫자.숫자.숫자
    r" - - \["                            # 고정 구분 부분
    r"(?P<ts>[^\]]+)"                     # 시각: ] 가 아닌 글자들
    r"\] \""                              # ] 와 따옴표
    r"(?P<method>\w+) "                   # 방식: GET/POST
    r"(?P<path>\S+)"                      # 경로: 공백 아닌 글자들
    r"[^\"]*\" "                          # HTTP/1.1 부분 건너뛰기
    r"(?P<status>\d+) "                   # 상태 코드
    r"(?P<size>\d+)"                      # 응답 크기
)
logs = web_logs["raw"].str.extract(pattern)
print("추출된 표:", logs.shape)
print(logs.head())
print("\n추출 실패(어느 한 칸이라도 NaN) 행 수:", logs.isna().any(axis=1).sum())

In [ ]:
# 시나리오 2 — 타입 정리(날짜·숫자) + dt 부품
logs["ts"] = pd.to_datetime(logs["ts"], format="%d/%b/%Y:%H:%M:%S", errors="coerce")
logs["status"] = logs["status"].astype(int)
logs["size"] = logs["size"].astype(int)

# dt 부품: 시각(hour)과 요일
logs["hour"] = logs["ts"].dt.hour
logs["dow_name"] = logs["ts"].dt.day_name()

print("정리 후 자료형:")
print(logs.dtypes)
print()
print(logs[["ip", "ts", "method", "path", "status", "hour", "dow_name"]].head())

In [ ]:
# 시나리오 2 — 상태 코드·인기 경로 빠르게 집계
print("[상태 코드 분포]")
print(logs["status"].value_counts())

error_rate = (logs["status"] >= 400).mean() * 100   # 4xx·5xx 비율
print(f"\n오류 응답(4xx·5xx) 비율: {error_rate:.1f}%")

print("\n[요청이 많은 경로 TOP 5]")
print(logs["path"].value_counts().head())

In [ ]:
# 시나리오 3 — 시간대별 트래픽 (resample 'h')
log_ts = logs.dropna(subset=["ts"]).set_index("ts").sort_index()
hourly_traffic = log_ts["ip"].resample("h").count()    # 시간당 요청 수

plt.figure(figsize=(12, 4))
plt.plot(hourly_traffic.index, hourly_traffic.values, color="#4C72B0")
plt.title("Hourly Web Traffic (requests per hour)")
plt.xlabel("time"); plt.ylabel("requests")
plt.tight_layout(); plt.show()

print("가장 트래픽이 많았던 시간대 TOP 3:")
print(hourly_traffic.sort_values(ascending=False).head(3))

In [ ]:
# 코드 퀴즈 — 모범 답안
# 1) 정제(strip+lower) → 2) @ 뒤 도메인 추출 → 3) 조건 집계
clean = customers["email"].str.strip().str.lower()
domain = clean.str.split("@").str[1]
gmail_count = (domain == "gmail.com").sum()

print("gmail.com 고객 수:", gmail_count)